# Futures — Multilenguaje (Python, C++, Java, C#, Rust)

**Objetivo:** lanzar tareas y recolectar resultados con *futures*.
- Un **Future** representa un **resultado pendiente**.
- **I/O-bound**: futures + thread pools → buena **concurrencia**.
- **CPU-bound**: paralelismo real según runtime/SO; en CPython usar **processes**.


**¿Qué es un Future?**

Un Future (futuro) es básicamente un placeholder para un resultado que aún no está disponible en el presente, pero lo estará en el futuro producto de una computación asíncrona. En otras palabras, un Future representa el resultado pendiente de una tarea que se está ejecutando concurrentemente. Permite consultar si la tarea terminó, esperar a su finalización (bloquear o await hasta que esté lista), y luego obtener el resultado. Ejemplo: en Java, al someter una tarea a un ExecutorService, se obtiene un Future inmediato; el cálculo se ejecuta en otro hilo y eventualmente future.get() devolverá el resultado. En Python, las funciones de concurrent.futures y las Tasks de asyncio también producen objetos Future. En C# los Future se representan con Task<T>. En resumen: un Future desacopla el inicio de una tarea de la obtención de su resultado, facilitando la coordinación de múltiples tareas concurrentes.

**Uso típico de futuros:** Los futuros se usan para:

- **Lanzar tareas concurrentes sin bloquear:** obtienes un Future inmediatamente y puedes realizar otras operaciones mientras la tarea se procesa en segundo plano.

- **Sincronizar resultados:** más adelante, puedes esperar (join/await) a que el Future termine para recuperar el resultado, sincronizando ese punto en el programa.

- **Componer tareas:** algunos frameworks permiten encadenar futuros (por ej., CompletableFuture en Java, Task.ContinueWith en C#, combinators en JavaScript Promises, etc.), de modo que cuando un Future termina, desencadene automáticamente otra acción. Esto es útil en pipelines de procesamiento asíncrono.

- **Manejar timeouts o cancelación:** muchas implementaciones de Future soportan cancelar la tarea si ya no se necesita, o establecer un tiempo máximo de espera.


## 0) Instalación de herramientas (si no corriste antes)

In [1]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq g++ openjdk-17-jdk mono-devel
command -v rustc >/dev/null 2>&1 || (curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y)
source $HOME/.cargo/env || true
echo 'Listo.'


Selecting previously unselected package mono-runtime-common.
(Reading database ... 126435 files and directories currently installed.)
Preparing to unpack .../0-mono-runtime-common_6.8.0.105+dfsg-3.2_amd64.deb ...
Unpacking mono-runtime-common (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-corlib4.5-dll.
Preparing to unpack .../1-libmono-corlib4.5-dll_6.8.0.105+dfsg-3.2_all.deb ...
Unpacking libmono-corlib4.5-dll (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-system-core4.0-cil.
Preparing to unpack .../2-libmono-system-core4.0-cil_6.8.0.105+dfsg-3.2_all.deb ...
Unpacking libmono-system-core4.0-cil (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-system-numerics4.0-cil.
Preparing to unpack .../3-libmono-system-numerics4.0-cil_6.8.0.105+dfsg-3.2_all.deb ...
Unpacking libmono-system-numerics4.0-cil (6.8.0.105+dfsg-3.2) ...
Selecting previously unselected package libmono-system-xml4.0-cil.
Preparing to unpack .../4-

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
info: downloading installer
info: profile set to 'default'
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for 'stable-x86_64-unknown-linux-gnu'
info: latest update on 2025-08-07, rust version 1.89.0 (29483883e 2025-08-04)
info: downloading component 'cargo'
info: downloading component 'clippy'
info: downloading component 'rust-docs'
info: downloading component 'rust-std'
info: downloading component 'rustc'
info: downloading component 'rustfmt'
info: installing component 'cargo'
info: installing component 'clippy'
info: installing component 'rust-docs'
info: installing component 'rust-std'
info: installing component 'rustc'
info: installing component 'rustfmt'
info: default toolchain set to 'stable-x86_64-unknown-linux-gnu'


## 1) Python — `concurrent.futures`

In [2]:
import time, concurrent.futures as cf
def io_task(lat=0.08): time.sleep(lat); return 1
print("ThreadPool I/O...")
t0=time.perf_counter()
with cf.ThreadPoolExecutor(max_workers=20) as ex:
    futs=[ex.submit(io_task) for _ in range(40)]
    total = sum(f.result() for f in cf.as_completed(futs))
print("Total:", total, "| tiempo:", time.perf_counter()-t0, "seg")

def cpu_task(n=800_000):
    s=0
    for i in range(n): s+=i*i
    return s

print("ProcessPool CPU...")
t0=time.perf_counter()
with cf.ProcessPoolExecutor() as ex:
    futs=[ex.submit(cpu_task) for _ in range(4)]
    _=[f.result() for f in futs]
print("Tiempo:", time.perf_counter()-t0, "seg")

ThreadPool I/O...
Total: 40 | tiempo: 0.1674856780000482 seg
ProcessPool CPU...
Tiempo: 0.1903898839999556 seg


## 2) C++ — `std::future` + `std::async`

In [3]:
%%bash
cat > fut.cpp << 'CPP'
#include <future>
#include <iostream>
#include <thread>
#include <chrono>
int pesado(int n){ std::this_thread::sleep_for(std::chrono::milliseconds(200)); return n*n; }
int main()
{
  auto f1 = std::async(std::launch::async, pesado, 5);
  auto f2 = std::async(std::launch::async, pesado, 7);
  std::cout << (f1.get() + f2.get()) << std::endl;
}
CPP
g++ -std=gnu++17 fut.cpp -O2 -lpthread -o fut
time ./fut


74



real	0m0.206s
user	0m0.002s
sys	0m0.001s


## 3) Java — `CompletableFuture`

In [4]:
%%bash
cat > Fut.java << 'JAVA'
import java.util.concurrent.*;
public class Fut
{
  static void sleep(long ms){ try{ Thread.sleep(ms);}catch(Exception e){} }
  public static void main(String[] a) throws Exception
  {
    var f1 = CompletableFuture.supplyAsync(() -> { sleep(200); return 25; });
    var f2 = CompletableFuture.supplyAsync(() -> { sleep(200); return 49; });
    System.out.println(f1.thenCombine(f2, Integer::sum).get());
  }
}
JAVA
javac Fut.java
time java Fut


74



real	0m0.269s
user	0m0.053s
sys	0m0.030s


## 4) C# — `Task` como futuro

In [7]:
%%bash
cat > F.cs << 'CS'
using System;
using System.Threading.Tasks;
using System.Diagnostics;

class P
{
  static int Pesado(int x)
  {
    System.Threading.Thread.Sleep(200); // simula CPU o I/O bloqueante
    return x*x;
  }

  // Lógica asíncrona real
  static async Task<int[]> RunAsync()
  {
    var t0 = Stopwatch.StartNew();

    var t1 = Task.Run(() => Pesado(5));
    var t2 = Task.Run(() => Pesado(7));

    var r = await Task.WhenAll(t1, t2);

    t0.Stop();
    Console.WriteLine(r[0] + r[1]);
    Console.WriteLine("Tiempo: " + t0.ElapsedMilliseconds + " ms");

    return r;
  }

  // Punto de entrada compatible con mcs/Mono (.NET clásico)
  public static void Main(string[] args)
  {
    RunAsync().GetAwaiter().GetResult();
  }
}
CS
mcs -langversion:latest F.cs -out:f.exe
time mono f.exe


74
Tiempo: 206 ms



real	0m0.231s
user	0m0.024s
sys	0m0.011s


## 5) Rust — `futures`/Tokio

In [12]:
%%bash
set -e
source $HOME/.cargo/env
cargo new --bin fut_demo >/dev/null 2>&1 || true
cd fut_demo
cat > Cargo.toml << 'TOML'
[package]
name = "fut_demo"
version = "0.1.0"
edition = "2021"

[dependencies]
tokio = { version = "1", features = ["full"] }
futures = "0.3"
TOML
cat > src/main.rs << 'RS'
use tokio::time::{sleep, Duration};
use futures::future::{join, join_all};
#[tokio::main]
async fn main()
{
    let (a,b) = join(async { sleep(Duration::from_millis(150)).await; 3 }, async { sleep(Duration::from_millis(150)).await; 4 }).await;
    println!("{}", a+b);
    let tasks = (1..=3).map(|i| async move { sleep(Duration::from_millis(50*i)).await; i*i });
    let v = join_all(tasks).await;
    println!("{:?}", v);
}
RS
time cargo run --quiet


7
[1, 4, 9]



real	0m1.509s
user	0m0.594s
sys	0m0.619s


### Conclusión
- **Futures** coordinan resultados pendientes.
- El **paralelismo** depende del *executor* y del SO; en CPython, CPU-bound ⇒ `ProcessPool`.
